# Presenza AI - Model Fine-Tuning
This Colab Notebook pulls face datasets from Supabase, processes embeddings using `face_recognition`, and updates the Supabase Postgres database.

In [ ]:
!pip install supabase python-dotenv face_recognition opencv-python numpy

In [ ]:
import os
import numpy as np
import cv2
import face_recognition
from supabase import create_client, Client
from google.colab import userdata

# Fetch secrets from Google Colab Secrets (Left sidebar > Secrets)
SUPABASE_URL = userdata.get('SUPABASE_URL')
SUPABASE_SERVICE_KEY = userdata.get('SUPABASE_SERVICE_KEY')

if not SUPABASE_URL or not SUPABASE_SERVICE_KEY:
    raise ValueError("Missing Supabase credentials. Add them to Colab Secrets.")

supabase: Client = create_client(SUPABASE_URL, SUPABASE_SERVICE_KEY)
print("Connected to Supabase!")


In [ ]:
def normalize_vector(vector):
    norm = np.linalg.norm(vector)
    if norm == 0:
        return vector
    return vector / norm

bucket = "dataset"
files_response = supabase.storage.from_(bucket).list()

if not files_response:
    print("No folders found in dataset bucket.")
else:
    print(f"Found folders: {[f['name'] for f in files_response]}")

    for folder in files_response:
        user_id = folder["name"]
        if not user_id.isdigit():
            continue
            
        print(f"\nProcessing User ID: {user_id}")
        user_files = supabase.storage.from_(bucket).list(user_id)
        
        embeddings = []
        for f in user_files:
            file_name = f['name']
            if not file_name.endswith('.jpg'): continue
            
            file_path = f"{user_id}/{file_name}"
            res = supabase.storage.from_(bucket).download(file_path)
            
            # Decode image from bytes
            nparr = np.frombuffer(res, np.uint8)
            img = cv2.imdecode(nparr, cv2.IMREAD_COLOR)
            rgb_img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            
            # Extract embedding
            encodings = face_recognition.face_encodings(rgb_img)
            if encodings:
                embeddings.append(encodings[0])
                print(f"  - Extracted face from {file_name}")
            else:
                print(f"  - No face found in {file_name}")
                
        if embeddings:
            # Compute average embedding for robustness
            avg_embedding = np.mean(embeddings, axis=0)
            avg_embedding = normalize_vector(avg_embedding)
            
            # Update database record
            response = supabase.table("face_profiles").update({"embedding": avg_embedding.tolist()}).eq("user_id", int(user_id)).execute()
            
            if response.data:
                print(f"✅ Successfully updated embedding for user {user_id} in database.")
            else:
                print(f"❌ Failed to update embedding for user {user_id}.")
        else:
            print(f"⚠️ No embeddings extracted for user {user_id}.")
